# 🎯 [DAT06_M07] 결산 보고서 최종 발간 전 군수과장/보급관 3단계 HITL 전자 결재선 연계 설계
### Google Colab 실습 환경 (데이터·자동화·업무플랫폼 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/DAT06/dat06_m07_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 데이터·자동화·업무플랫폼 (Level 3) |
| **미션 ID** | DAT06_M07 |
| **미션 주제** | 결산 보고서 최종 발간 전 군수과장/보급관 3단계 HITL 전자 결재선 연계 설계 |
| **소요 시간** | 약 3.0시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
미국 국방재무회계국(DFAS)은 반복적인 재무·정산 업무에 업무 자동화(RPA)를 도입해 사람이 하던 취합과 점검을 줄이는 사업을 추진해 왔다. 이 공개 사례의 학습 초점은 반복 업무 분해, 입력과 트리거, 자동 처리 순서, 오류 복구, 실행 로그, 시간 절감이다. 본 프로젝트는 이를 바탕으로 교육용 가상 부대 24개의 월간 결산 자료(물자 불출, 수리부속 소모, 탄약 정산, 장비 가동률)를 코드로 만들고, 파일 대신 메모리 안의 가상 문서와 표로 모의한다. 학습자는 업무 분해, 다형식 파싱, 검증 룰 엔진, 이상 원인 추천, 일관성 검증, 승인·반려 대시보드, 3단계 사람 결재선, 대량 배치 시험, AAR 보정, 최종 통합을 차례로 수행한다. 최종 목표는 월간 정산·보고·취합을 한 번에 실행하되 발간은 반드시 사람이 결재하도록 하는 원클릭 자동화 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> 💡 **[산출물(Deliverables) 디렉토리 경로 및 실습 안내]**
> - **아키텍처 관점의 표준 산출물:** 과제 명세에 표기된 src/connectors/..., docs/..., dist/... 등의 경로는 본 솔루션이 실무 배포 환경에서 갖추게 되는 **소프트웨어 표준 아키텍처 구조**를 의미합니다.
> - **노트북 중심 실습:** 별도의 폴더나 외부 파이썬 파일(.py)을 수작업으로 생성하실 필요가 없습니다. 본 노트북의 각 미션별 실습 셀을 AI 코파일럿과 함께 완성하고 검증하면 해당 산출물 요건이 자동으로 충족됩니다.
> - **미션 10 최종 통합 패키징:** 각 미션 셀에서 구현된 단위 모듈과 명세는 **[미션 10]** 실행 시 단일 소프트웨어 패키지(dist/dat06_package/)와 운영 매뉴얼(docs/dat06_final_manual.md)로 일괄 결합되어 자동 추출됩니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 07. 결산 보고서 최종 발간 전 군수과장/보급관 3단계 HITL 전자 결재선 연계 설계
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

결산 보고서 최종 발간 전 군수과장/보급관 3단계 HITL 결재선을 설계한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **결산 보고서 발간 전 담당자 기안, 보급관 검토, 군수과장 승인의 3단계 사람 결재선을 설계하고 이력의 무결성과 권한 통제를 검증한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **결재 단계별 역할과 순서를 설정으로 고정하고, 순서 위반, 본인 결재, 근거 미기재, 미등록 사용자를 차단 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/workflow/dat06_approval_system.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군 보안 규정 및 행정 절차 부합성

#### 미션 PASS 조건

- 정량 기준: 이력 기록 무결성 100%, 승인 권한 우회 시도 차단율 100%


- 진행 기준: 결재선 승인/반려 시뮬레이션 및 이력 조회 화면 구현


- 검증 방법: 권한별 접근 제어(RBAC) 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""결산 보고서 발간 전 담당자 기안, 보급관 검토, 군수과장 최종 승인의 3단계 사람 결재선을 설계하고, 해시 체인 감사 이력의 무결성과 권한 우회 시도 10종의 차단을 검증한다. 자동 발간은 허용하지 않는다."""
import hashlib
import json
import numpy as np
import pandas as pd

# [교육용가상자료] 가상 직책·인물(가명). 실제 전자 결재 체계와 무관한 학습용 모의
USERS = {"kim": "담당자", "lee": "담당자", "park": "보급관", "choi": "군수과장", "han": "열람자"}
LINE = [("기안", "담당자"), ("검토", "보급관"), ("승인", "군수과장")]       # 순서 고정 3단계
AUDIT = []


def _h(prev, rec):
    return hashlib.sha256((prev + json.dumps(rec, ensure_ascii=False, sort_keys=True)).encode()).hexdigest()[:16]


def log(doc, user, action, result, note=""):
    rec = {"seq": len(AUDIT) + 1, "doc": doc, "user": user, "role": USERS.get(user, "미등록"), "action": action, "result": result, "note": note}
    rec["hash"] = _h(AUDIT[-1]["hash"] if AUDIT else "GENESIS", {k: v for k, v in rec.items() if k != "hash"})
    AUDIT.append(rec)


def verify_chain():
    prev, ok = "GENESIS", 0
    for r in AUDIT:
        body = {k: v for k, v in r.items() if k != "hash"}
        ok += (r["hash"] == _h(prev, body)); prev = r["hash"]
    return ok, len(AUDIT)


DOCS = {}


def step(doc, user, action, comment=""):
    """결재 처리. 권한·순서·본인 결재 금지·근거 기재를 모두 통과해야 기록되고, 실패는 차단 기록을 남긴다."""
    d = DOCS.setdefault(doc, {"stage": 0, "signers": [], "published": False})
    role = USERS.get(user)
    if role is None:
        return log(doc, user, action, "차단", "미등록 사용자"), False
    if d["stage"] >= len(LINE):
        return log(doc, user, action, "차단", "이미 결재 완료"), False
    need_act, need_role = LINE[d["stage"]]
    if action != need_act:
        return log(doc, user, action, "차단", f"순서 위반: 현재 단계 '{need_act}'"), False
    if role != need_role:
        return log(doc, user, action, "차단", f"권한 없음: 역할 '{role}'은(는) '{action}' 불가({need_role} 필요)"), False
    if user in d["signers"]:
        return log(doc, user, action, "차단", "본인이 앞 단계에 서명한 문서는 다음 단계를 결재할 수 없음"), False
    if action != "기안" and len(comment.strip()) < 5:
        return log(doc, user, action, "차단", "검토 근거(5자 이상) 미기재"), False
    d["stage"] += 1; d["signers"].append(user)
    log(doc, user, action, "승인", comment or "기안 상신")
    return None, True


def publish(doc, user):
    d = DOCS.get(doc)
    if not d or d["stage"] < len(LINE):
        log(doc, user, "발간", "차단", "3단계 결재 미완료(자동 발간 금지)"); return False
    if USERS.get(user) != "담당자":
        log(doc, user, "발간", "차단", "발간은 담당자가 결재 완료 후 수동 실행"); return False
    d["published"] = True; log(doc, user, "발간", "승인", "결재 3단계 완료 후 수동 발간"); return True


# 1) 정상 결재 흐름 2건
for doc, drafter in (("RPT-2026-08-A", "kim"), ("RPT-2026-08-B", "lee")):
    step(doc, drafter, "기안")
    step(doc, "park", "검토", "재고 산식 및 전월 연속성 확인 완료")
    step(doc, "choi", "승인", "보급관 검토 의견 확인, 발간 승인")
    publish(doc, drafter)

# 2) 권한 우회 시도 10종 - 결재가 진행되는 순서에 맞춰 시도하고, 모두 차단되어야 한다
blocked, total = 0, 0
print("[권한 우회 시도 결과] [교육용가상자료]")


def attempt(name, fn):
    global blocked, total
    n = len(AUDIT); fn(); r = AUDIT[-1]
    ok = len(AUDIT) == n + 1 and r["result"] == "차단"
    blocked += ok; total += 1
    print(f" - {name}: {r['result']} ({r['note']})")


attempt("미등록 사용자 기안", lambda: step("RPT-ATT", "hacker", "기안"))
attempt("열람자가 기안", lambda: step("RPT-ATT", "han", "기안"))
attempt("기안 전에 군수과장이 승인(단계 건너뛰기)", lambda: step("RPT-ATT", "choi", "승인", "바로 승인합니다 확인"))
step("RPT-ATT", "kim", "기안")
attempt("기안자 본인이 검토까지 처리", lambda: step("RPT-ATT", "kim", "검토", "본인이 검토합니다 확인"))
attempt("다른 담당자가 검토 단계 처리", lambda: step("RPT-ATT", "lee", "검토", "다른 담당자 검토 시도 확인"))
attempt("검토 근거 없이 보급관 검토", lambda: step("RPT-ATT", "park", "검토", ""))
attempt("보급관 검토 전에 군수과장이 승인", lambda: step("RPT-ATT", "choi", "승인", "보급관 단계를 생략한 승인 확인"))
attempt("결재 미완료 상태에서 발간", lambda: publish("RPT-ATT", "kim"))
step("RPT-ATT", "park", "검토", "산식 검증 결과 이상 없음 확인")
step("RPT-ATT", "choi", "승인", "최종 승인합니다 확인")
attempt("완결된 문서 재승인 시도", lambda: step("RPT-ATT", "choi", "승인", "한 번 더 승인 시도 확인"))
attempt("보급관이 발간 시도", lambda: publish("RPT-ATT", "park"))
print(f"우회 시도 {total}건 중 차단 {blocked}건({blocked / total:.0%})")

# 3) 이력 무결성: 정상 체인 검증 후 위·변조 시험
ok, tot = verify_chain()
print(f"\n이력 {tot}건 해시 체인 검증: {ok}/{tot} 일치({ok / tot:.0%})")
k = next(i for i, r in enumerate(AUDIT) if r["result"] == "차단")
saved = AUDIT[k]["result"]
AUDIT[k]["result"] = "승인"          # 차단 기록을 승인으로 위조
ok2, _ = verify_chain()
AUDIT[k]["result"] = saved
print(f"위조 시험: {k + 1}번 기록 변조 시 검증 {ok2}/{tot} -> 변조 탐지 {'성공' if ok2 < tot else '실패'}, 원복 후 {verify_chain()[0]}/{tot}")
st = pd.DataFrame(AUDIT).groupby("doc")["result"].value_counts().unstack(fill_value=0)
print("\n[문서별 결재 이력 요약]")
print(st.to_string())
pubs = [k for k, v in DOCS.items() if v["published"]]
print("발간 완료 문서:", pubs, "/ 우회 시험 문서 RPT-ATT는 발간되지 않음 / 자동 발간 0건(발간은 담당자 수동 실행)")
print(f"목표(이력 무결성 100%, 우회 차단율 100%): {'달성' if ok == tot and blocked == total and ok2 < tot else '미달'}")
print("실제 결재와 보안 규정은 부대 지침과 지휘계통에 따른다. 본 코드는 학습용 모의이다.")


<details>
<summary><b>💡 [참고] 미션 07 예상 실행 결과 (Output)</b></summary>

```text
[권한 우회 시도 결과] [교육용가상자료]
 - 미등록 사용자 기안: 차단 (미등록 사용자)
 - 열람자가 기안: 차단 (권한 없음: 역할 '열람자'은(는) '기안' 불가(담당자 필요))
 - 기안 전에 군수과장이 승인(단계 건너뛰기): 차단 (순서 위반: 현재 단계 '기안')
 - 기안자 본인이 검토까지 처리: 차단 (권한 없음: 역할 '담당자'은(는) '검토' 불가(보급관 필요))
 - 다른 담당자가 검토 단계 처리: 차단 (권한 없음: 역할 '담당자'은(는) '검토' 불가(보급관 필요))
 - 검토 근거 없이 보급관 검토: 차단 (검토 근거(5자 이상) 미기재)
 - 보급관 검토 전에 군수과장이 승인: 차단 (순서 위반: 현재 단계 '검토')
 - 결재 미완료 상태에서 발간: 차단 (3단계 결재 미완료(자동 발간 금지))
 - 완결된 문서 재승인 시도: 차단 (이미 결재 완료)
 - 보급관이 발간 시도: 차단 (발간은 담당자가 결재 완료 후 수동 실행)
우회 시도 10건 중 차단 10건(100%)

이력 21건 해시 체인 검증: 21/21 일치(100%)
위조 시험: 9번 기록 변조 시 검증 20/21 -> 변조 탐지 성공, 원복 후 21/21

[문서별 결재 이력 요약]
result         승인  차단
doc                  
RPT-2026-08-A   4   0
RPT-2026-08-B   4   0
RPT-ATT         3  10
발간 완료 문서: ['RPT-2026-08-A', 'RPT-2026-08-B'] / 우회 시험 문서 RPT-ATT는 발간되지 않음 / 자동 발간 0건(발간은 담당자 수동 실행)
목표(이력 무결성 100%, 우회 차단율 100%): 달성
실제 결재와 보안 규정은 부대 지침과 지휘계통에 따른다. 본 코드는 학습용 모의이다.
```
</details>
